In [ ]:
# 유용원의 군사세계 사이트 데이터 크롤링 (WebDriverWait.until(...) "직접" 스타일 버전)
# ✅ time.sleep 최소화 / 요소 뜨면 바로 진행
# ✅ 네가 말한 wait = WebDriverWait(driver, 100) 형태로만 구성

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException
import time
import math
import pandas as pd
import os

from openpyxl.styles import Font
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
import chromedriver_autoinstaller

print('='*80)
print('< 이 프로그램은 유용원의 군사세계 사이트에서 각종 뉴스 데이터를 수집하는 프로그램 입니다.>')
print('='*80)

# =========================
# 입력
# =========================
q = input('검색하고 싶은 검색어는?(기본값: 우크라이나): ')
if q == '':
    q = '우크라이나'

qu = 'https://bemil.chosun.com/nbrd/bbs/list.html?b_bbs_id=10158'

cnt = input('원하시는 수집 건수는 몇건 입니까?(기본값: 20): ')
if cnt == '':
    cnt = 20
cnt = int(cnt)

pcnt = math.ceil(cnt / 15)

now = time.localtime()
stamp = '%04d-%02d-%02d-%02d-%02d-%02d' %(
    now.tm_year, now.tm_mon, now.tm_mday,
    now.tm_hour, now.tm_min, now.tm_sec
)

ff = 'c:\\py_temp\\'
if ff == '':
    ff = 'c:\\py_temp\\'

base_dir = ff + stamp + '-' + q + '\\'
img_dir  = base_dir + 'img'

fx_name = base_dir + stamp + '-' + q + '.xlsx'
fc_name = base_dir + stamp + '-' + q + '.csv'

os.makedirs(img_dir, exist_ok=True)

# =========================
# 드라이버
# =========================
chromedriver_autoinstaller.install(path='c:/py_temp')

options = webdriver.ChromeOptions()
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
driver.set_page_load_timeout(30)

# ✅ 네가 원하는 웨잇 양식
wait = WebDriverWait(driver, 100)

# =========================
# 사이트 접속
# =========================
driver.get(qu)
wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
driver.maximize_window()

# =========================
# 코너 선택
# =========================
cono = input(' 원하시는 코너가 있으습니까?(기본값: 전체) Y or N: ').upper()
if cono == 'Y':
    wait.until(
        EC.element_to_be_clickable((By.XPATH,'//*[@id="container-area"]/div[1]/div[2]/div[1]/section/div[2]'))
    ).click()

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    options_list = soup.find('div', 'select-group').find_all('option')

    print('선택가능한 사이트는 다음과 같습니다.\n')
    siteno = 1
    for opt in options_list:
        print(f"{siteno}. {opt.get_text()}")
        siteno += 1

    print()
    site = input('원하시는 사이트를 입력 하세요(기본값: 전체) 번호로 입력해줘용~~^^: \n')
    if site != '':
        wait.until(
            EC.element_to_be_clickable((By.XPATH, f'//*[@id="branchid"]/option[{int(site)}]'))
        ).click()

# 내용 찾을때
def scroll_down(driver):      
    driver.execute_script("window.scrollBy(0,400);")
    time.sleep(1)    

# 검색창 찾을때
def scroll_down2(driver):      
    driver.execute_script("window.scrollBy(0,500);")
    time.sleep(1)   
    
# =========================
# 검색
# =========================
try:
    scroll_down2(driver)
    search_xpath = '//*[@id="container-area"]/div[1]/div[2]/div[1]/section/div[3]/div[3]/span/form/input[1]'
    
    search = wait.until(
        EC.visibility_of_element_located((By.XPATH, search_xpath))
    )
    search.click()
    search.clear()
    search.send_keys(q)
    search.send_keys(Keys.ENTER)
except:
    scroll_down2(driver)
    search_xpath = '//*[@id="container-area"]/div[1]/div[2]/div[1]/section/div[3]/div[3]/span/form/input[1]'
    
    search = wait.until(
        EC.visibility_of_element_located((By.XPATH, search_xpath))
    )
    search.click()
    search.clear()
    search.send_keys(q)
    search.send_keys(Keys.ENTER)

# 검색 결과 테이블 뜰 때까지
wait.until(
    EC.presence_of_element_located((By.XPATH, '//*[@id="container-area"]/div[1]/div[2]/div[1]/section/div[3]/table'))
)

# =========================
# 결과 저장 리스트
# =========================
no2, title2, name2, date2, inquiry2, recom2, story2, link2 = ([] for _ in range(8))

no = 1
pcno = 1
stop_all = False

print()
print('데이터를 수집합니다\n')

# =========================
# 크롤링 루프
# =========================
for w in range(4, 14):
    if stop_all:
        break

    # 테이블 로드 대기
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "BasicTables")))

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    c1 = soup.find('table', 'BasicTables')
    if c1 is None:
        print("테이블을 찾지 못했습니다. 사이트 구조가 바뀌었을 수 있어요.")
        break

    story = []
    c2 = c1.find_all('td')
    for a in c2:
        story.append(a.get_text().strip())

    for e in range(0, 15):
        if no > cnt:
            stop_all = True
            break

        # 게시글 클릭 (두 형태 시도)
        row_xpath_1 = f'//*[@id="container-area"]/div[1]/div[2]/div[1]/section/div[3]/table/tbody/tr[{e+1}]/td[2]/a'
        row_xpath_2 = f'//*[@id="container-area"]/div[1]/div[2]/div[1]/section/div[3]/table/tbody/tr[{e+1}]/td[2]/a[1]'

        clicked = False

        # 최대 3번 시도: (클릭 → 스크롤 → 클릭 → 스크롤 → 클릭)
        for _ in range(3):
            try:
                try:
                    wait.until(
                        EC.element_to_be_clickable((By.XPATH, row_xpath_1))
                    ).click()
                except:
                    wait.until(
                        EC.element_to_be_clickable((By.XPATH, row_xpath_2))
                    ).click()
        
                # ✅ 본문이 뜨는 즉시 진행
                wait.until(
                    EC.presence_of_element_located((By.CLASS_NAME, "board_body"))
                )
                wait.until(
                    lambda d: d.execute_script("return document.readyState") == "complete"
                )
        
                clicked = True
                break  # 성공했으니 루프 탈출
        
            except:
                # 클릭 실패 → 스크롤 후 재시도
                scroll_down(driver)
        
        # 3번 다 실패 → 자료 자체가 없음 → 전체 종료
        if not clicked:
            print('자료가 전부입니다ㅠㅠ')
            stop_all = True
            break


        # 본문 파싱
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        board = soup.find('div', 'board_body')
        blocks = []
        if board:
            blocks = board.find_all('strong')
            if len(blocks) < 2:
                blocks = board.find_all('div')
            if len(blocks) < 2:
                blocks = board.find_all('p')

        text = ""
        for c in blocks:
            s1 = c.get_text().strip()
            if s1:
                text += s1 + "\n"

        clean_text = "\n".join([line for line in text.splitlines() if line.strip() != ""])

        print('-'*80)

        no2.append(str(no) + '\n')
        print('1. 번호: ' + str(no))

        # 제목/작성자/날짜/조회/추천
        try:
            title = c2[1+(e*6)].find('a').get_text()
        except:
            title = ""
        title2.append(title)
        print('2. 제목: ' + title)

        name = story[2+(e*6)] if len(story) > 2+(e*6) else ""
        name2.append(name)
        print('3. 작성자명: ' + name)

        date = story[3+(e*6)] if len(story) > 3+(e*6) else ""
        date2.append(date)
        print('4. 작성일: ' + date)

        inquiry = story[4+(e*6)] if len(story) > 4+(e*6) else ""
        inquiry2.append(inquiry + '\n')
        print('5. 조회수: ' + inquiry)

        recom = story[5+(e*6)] if len(story) > 5+(e*6) else ""
        recom2.append(recom + '\n')
        print('6. 추천수: ' + recom)

        story2.append(clean_text)
        print('7. 내용: ' + clean_text)

        # 링크 생성(원 코드 유지)
        num_val = story[0+(e*6)] if len(story) > 0+(e*6) else ""
        link = f'https://bemil.chosun.com/nbrd/bbs/view.html?b_bbs_id=10158&branch=&pn={w-3}&num={num_val}'
        link2.append(link)
        print('8. 컬럼 링크: ' + link + '\n')

        # 뒤로가기 후 리스트 테이블 복귀 대기
        driver.back()
        wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "BasicTables")))

        no += 1

    if stop_all:
        break

    pcno += 1
    if pcno > pcnt:
        break

    print('다음 페이지로 넘어갑니다~~\n')

    next_page_xpath = f'//*[@id="container-area"]/div[1]/div[2]/div[1]/section/div[3]/div[2]/a[{w}]'
    
    page_moved = False
    
    # 최대 3번 시도: 클릭 → 스크롤 → 클릭 → 스크롤 → 클릭
    for _ in range(3):
        try:
            wait.until(
                EC.element_to_be_clickable((By.XPATH, next_page_xpath))
            ).click()
    
            # 페이지 로드 대기
            wait.until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
            wait.until(
                EC.presence_of_element_located((By.CLASS_NAME, "BasicTables"))
            )
    
            page_moved = True
            break
    
        except:
            # 클릭 실패 → 스크롤 후 재시도
            scroll_down(driver)
    
    # 3번 다 실패 → 마지막 페이지
    if not page_moved:
        print('마지막 페이지 입니다ㅠㅠ')
        break


# =========================
# 저장
# =========================
bemil = pd.DataFrame({
    "번호": no2,
    "제목": pd.Series(title2),
    "작성자명": pd.Series(name2),
    "작성일": pd.Series(date2),
    "조회수": pd.Series(inquiry2),
    "추천수": pd.Series(recom2),
    "내용": pd.Series(story2),
    "링크": pd.Series(link2),
})

bemil.to_csv(fc_name, encoding="utf-8-sig", index=False)
bemil.to_excel(fx_name, index=False)

print('='*80)
print('저장을 완료하였습니다.')
print('='*80)

# =========================
# 엑셀 하이퍼링크
# =========================
wb = load_workbook(fx_name)
ws = wb["Sheet1"]

for row in range(2, ws.max_row + 1):
    cell = ws.cell(row=row, column=8)  # 8번째 열이 '링크'
    url = cell.value
    if isinstance(url, str) and url.startswith("http"):
        cell.hyperlink = url
        cell.style = "Hyperlink"
        cell.font = Font(color="0000EE", underline="single")

wb.save(fx_name)

# driver.close()
print('='*80)
print('작업을 완료하였습니다. ^_^')
print('='*80)


In [1]:
from konlpy.tag import *        #pip install konlpy 먼저 하세요
import matplotlib.pyplot as plt #pip install matplotlib 먼저 하세요
from matplotlib import font_manager , rc
from wordcloud import WordCloud  # pip install wordcloud 먼저 하세요
from collections import Counter
from PIL import Image      # pip install Image
from wordcloud import ImageColorGenerator
import nltk
from nltk.probability import FreqDist

In [5]:
data = pd.read_excel('C:\\py_temp\\2025-12-31-12-00-17-우크라이나\\2025-12-31-12-00-17-우크라이나.xlsx')
data1 = data[data['작성일'] >= '2023.02.23']['내용']
data1

okt = Okt()
kkma = Kkma( )

data3 = []
for a in range(0, len(data1)):
    data2 = okt.nouns(data1[a])
    for i in range(0, len(data2)):
        data3.append(data2[i])


In [ ]:
sword = open("c:\\py_temp\\불용어목록.txt", encoding = 'utf-8').read()

data4 = [ each_word for each_word in data3
          if each_word not in sword ]

data5 = Counter(data4)
print(data5)
data6 = data5.most_common(20)
data7 = dict(data6)
data7

wordcloud = WordCloud(font_path="c:\\windows\\fonts\\gulim.TTc" ,
                       relative_scaling=0.9,
                       background_color="white"
                     ).generate_from_frequencies(data7)
plt.figure(figsize=(8,4))
plt.imshow(wordcloud)
plt.axis('off')
plt.show()